# 🏦 CashGuard — Дообучение Qwen2.5-VL-7B-Instruct (LoRA / QLoRA)
### ForteBank Datathon 2026 | AI Аудит кассовой дисциплины

> **Цель**: Дообучить Vision-Language модель `Qwen2.5-VL-7B-Instruct` выявлять нарушения кассовой дисциплины (согласно 10 правилам НБРК) с привязкой к статьям и таймкодам.
> **Среда**: Google Colab (бесплатный T4 GPU 16GB VRAM или A100/V100).

## 1. Проверка GPU

In [ ]:
!nvidia-smi

## 2. Установка LLaMA-Factory и зависимостей

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]"
!pip install bitsandbytes accelerate peft transformers torchvision qwen-vl-utils

## 3. Загрузка обучающего датасета
Загрузите сгенерированные файлы `sharegpt_train.json` и `sharegpt_val.json` из вашей папки `ml/dataset_output/`.

In [ ]:
import os
import json
from google.colab import files

os.makedirs("/content/LLaMA-Factory/data", exist_ok=True)
print("Пожалуйста, загрузите sharegpt_train.json и sharegpt_val.json:")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = os.path.join("/content/LLaMA-Factory/data", filename)
    with open(dest, "wb") as f:
        f.write(uploaded[filename])
    print(f"✔ Сохранён: {dest}")

## 4. Регистрация датасета в dataset_info.json

In [ ]:
dataset_info_path = "/content/LLaMA-Factory/data/dataset_info.json"

if os.path.exists(dataset_info_path):
    with open(dataset_info_path, "r", encoding="utf-8") as f:
        info = json.load(f)
else:
    info = {}

info["cashguard_train"] = {
    "file_name": "sharegpt_train.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"},
    "tags": {"role_tag": "from", "content_tag": "value", "user_tag": "human", "assistant_tag": "gpt"}
}
info["cashguard_val"] = {
    "file_name": "sharegpt_val.json",
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"},
    "tags": {"role_tag": "from", "content_tag": "value", "user_tag": "human", "assistant_tag": "gpt"}
}

with open(dataset_info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print("✔ Датасет зарегистрирован в dataset_info.json")

## 5. Создание конфигурации обучения (QLoRA 4-bit)
Конфиг полностью оптимизирован под 16 ГБ VRAM (Google Colab T4).

In [ ]:
config_yaml = """
### Модель
model_name_or_path: Qwen/Qwen2.5-VL-7B-Instruct
trust_remote_code: true

### Метод (QLoRA)
stage: sft
do_train: true
finetuning_type: lora
quantization_bit: 4
quantization_method: bitsandbytes
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target: all

### Датасет
dataset: cashguard_train
eval_dataset: cashguard_val
template: qwen2_vl
cutoff_len: 2048
preprocessing_num_workers: 2

### Параметры обучения
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 1.0e-4
lr_scheduler_type: cosine
warmup_ratio: 0.1
num_train_epochs: 3
weight_decay: 0.01
max_grad_norm: 1.0

### Память
fp16: true
gradient_checkpointing: true
flash_attn: auto

### Логирование и выход
logging_steps: 5
eval_steps: 25
save_steps: 50
save_total_limit: 2
output_dir: /content/lora_output
do_eval: true
eval_strategy: steps
per_device_eval_batch_size: 1
seed: 42
"""

with open("/content/LLaMA-Factory/train_qwen_vl.yaml", "w", encoding="utf-8") as f:
    f.write(config_yaml.strip())
print("✔ train_qwen_vl.yaml сохранён")

## 6. Запуск процесса дообучения
Процесс займет около 15–25 минут на GPU T4.

In [ ]:
!llamafactory-cli train /content/LLaMA-Factory/train_qwen_vl.yaml

## 7. Скачивание полученного LoRA-адаптера
Архивируем веса адаптера и скачиваем их. Затем распакуйте их в локальную папку `ml/lora_output/` вашего проекта CashGuard.

In [ ]:
!zip -r /content/cashguard_lora_output.zip /content/lora_output
from google.colab import files
files.download("/content/cashguard_lora_output.zip")
print("✔ Загрузка архива cashguard_lora_output.zip начата!")